# Pandas and PySpark Integration Patterns

PySpark offers several ways to bridge **pandas** and **Spark**.  Choosing
the right pattern depends on data size, operation complexity, and whether
you need distributed execution.

This notebook covers:
1. Spark → pandas (`toPandas`)
2. pandas → Spark (`createDataFrame`)
3. Pandas API on Spark (`pyspark.pandas`)
4. Pandas UDFs (`@pandas_udf`)
5. Real-world feature engineering and ML preprocessing

In [ ]:
import os
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

import numpy as np
import pandas as pd
import pyspark.pandas as ps
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

from spp.session import create_spark_session

spark = create_spark_session("notebook-pandas-integration")
print(f"Spark {spark.version}")

## Pattern 1: Spark → pandas (toPandas)

Aggregate or filter in Spark first, then pull the **small** result to
the driver as a `pd.DataFrame` for plotting, reporting, or local
analysis.  Arrow makes the transfer fast and columnar.

In [ ]:
df = spark.createDataFrame(
    [
        ("North", "Alice", 1200.0),
        ("North", "Bob", 800.0),
        ("South", "Carol", 1500.0),
        ("South", "Dave", 950.0),
        ("East", "Eve", 1100.0),
    ],
    ["region", "name", "revenue"],
)

summary = df.groupBy("region").agg(
    F.sum("revenue").alias("total_revenue"),
    F.count("name").alias("headcount"),
)

pdf = summary.toPandas()
print(type(pdf))
pdf

## Pattern 2: pandas → Spark (createDataFrame)

Start with a local `pd.DataFrame` — a lookup table, config data, or
small dataset — and push it into Spark for distributed processing.
With Arrow enabled the conversion is zero-copy where possible.

In [ ]:
pdf = pd.DataFrame({
    "user_id": range(1, 6),
    "signup_date": pd.date_range("2024-01-01", periods=5),
    "plan": ["free", "pro", "free", "enterprise", "pro"],
})
print("pandas input:")
print(pdf)

sdf = spark.createDataFrame(pdf)
print("\nSpark schema:")
sdf.printSchema()
sdf.show()

## Pattern 3: Pandas API on Spark

`pyspark.pandas` (aliased as `ps`) lets you write **pandas syntax**
that executes on Spark.  No data is pulled to the driver — everything
runs distributed.  Perfect when your team knows pandas but the data
is too large for a single machine.

In [ ]:
psdf = ps.DataFrame({
    "category": ["A", "B", "A", "C", "B", "A"],
    "amount": [100, 200, 150, 300, 250, 175],
})

print("GroupBy mean:")
print(psdf.groupby("category").mean())

print("\nFiltered (amount > 150):")
print(psdf[psdf["amount"] > 150])

# Convert to Spark DataFrame when you need Spark operations
sdf = psdf.to_spark()
print(f"\nSpark row count: {sdf.count()}")

## Pattern 4: Pandas UDFs

A `@pandas_udf` applies a **vectorised** function column-wise using
Arrow for data transfer.  Much faster than row-at-a-time Python UDFs
and ideal for custom transformations that Spark SQL doesn't support
natively.

In [ ]:
df = spark.createDataFrame(
    [(1, 10.0), (2, 20.0), (3, 30.0), (4, 40.0), (5, 50.0)],
    ["id", "value"],
)


@pandas_udf(DoubleType())
def normalize(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


result = df.withColumn("normalized", normalize("value"))
result.show()

## Real-World: Feature Engineering

A common pattern: aggregate in Spark, then pull the compact summary
to pandas for correlation analysis, plotting, or export.

In [ ]:
sales = spark.createDataFrame(
    [
        ("North", "Alice", 1200.0),
        ("North", "Bob", 800.0),
        ("South", "Carol", 1500.0),
        ("South", "Dave", 950.0),
        ("East", "Eve", 1100.0),
    ],
    ["region", "name", "revenue"],
)

summary = sales.groupBy("region").agg(
    F.sum("revenue").alias("total_revenue"),
    F.count("name").alias("headcount"),
)

pdf = summary.toPandas()
pdf["revenue_per_head"] = pdf["total_revenue"] / pdf["headcount"]
print("Feature-engineered summary:")
print(pdf)
print(f"\nCorrelation (revenue vs headcount): {pdf['total_revenue'].corr(pdf['headcount']):.4f}")

## Real-World: Group Feature Engineering

Use `groupBy().applyInPandas()` to compute **rolling windows**, **lag
features**, and **cumulative statistics** per entity — operations that
are natural in pandas but distributed across the cluster by Spark.

In [ ]:
ts_data = spark.createDataFrame(
    [
        (1, 1, 10.0), (1, 2, 20.0), (1, 3, 30.0), (1, 4, 25.0),
        (2, 1, 5.0),  (2, 2, 15.0), (2, 3, 10.0), (2, 4, 20.0),
    ],
    ["entity_id", "ts", "value"],
)


def engineer_features(pdf: pd.DataFrame) -> pd.DataFrame:
    """Add rolling mean, lag, and cumulative sum features per entity."""
    pdf = pdf.sort_values("ts").copy()
    pdf["rolling_mean_3"] = pdf["value"].rolling(window=3, min_periods=1).mean()
    pdf["lag_1"] = pdf["value"].shift(1)
    pdf["cumsum"] = pdf["value"].cumsum()
    return pdf


feature_schema = (
    "entity_id: long, ts: long, value: double, "
    "rolling_mean_3: double, lag_1: double, cumsum: double"
)

ts_data.groupBy("entity_id").applyInPandas(
    engineer_features, schema=feature_schema
).show()

## Real-World: ML Preprocessing

Per-group preprocessing with `applyInPandas` — fill missing values
with the group median, remove outliers beyond 3σ, and min-max scale
features, all within each group independently.

In [ ]:
raw = spark.createDataFrame(
    [
        (1, 10.0, 1.0),
        (1, 20.0, 2.0),
        (1, None, 3.0),
        (1, 15.0, None),
        (1, 100.0, 5.0),
        (2, 5.0, 10.0),
        (2, 8.0, 12.0),
        (2, None, 11.0),
        (2, 6.0, None),
    ],
    ["group_id", "feature_a", "feature_b"],
)


def full_preprocess(pdf: pd.DataFrame) -> pd.DataFrame:
    """Combined pipeline: fill → remove outliers → scale."""
    pdf = pdf.copy()

    # 1. Fill missing with group median
    for col in ["feature_a", "feature_b"]:
        pdf[col] = pdf[col].fillna(pdf[col].median())

    # 2. Remove outliers (3-sigma)
    mean = pdf["feature_a"].mean()
    std = pdf["feature_a"].std() or 1.0
    pdf = pdf[(pdf["feature_a"] - mean).abs() / std < 3]

    # 3. Min-max scale
    fmin = pdf["feature_a"].min()
    fmax = pdf["feature_a"].max()
    span = fmax - fmin or 1.0
    pdf["scaled_a"] = (pdf["feature_a"] - fmin) / span

    return pdf


preprocess_schema = (
    "group_id: long, feature_a: double, feature_b: double, scaled_a: double"
)

raw.groupBy("group_id").applyInPandas(
    full_preprocess, schema=preprocess_schema
).show()

## Common Pitfalls

- **Calling `toPandas()` on large data** — always aggregate or filter
  in Spark first; `toPandas()` pulls everything to driver memory.
- **Mixing `pyspark.pandas` and `pandas` DataFrames** — convert
  explicitly with `ps.from_pandas()` or `.to_pandas()` to avoid
  confusing errors.
- **Forgetting Arrow** — without
  `spark.sql.execution.arrow.pyspark.enabled = true`, pandas
  conversions fall back to slow row-by-row serialisation.
- **Schema mismatch in `applyInPandas`** — the schema string must
  match the columns your function returns exactly.
- **Using regular Python UDFs** — prefer `@pandas_udf` for 10–100×
  better performance via Arrow vectorisation.

## Decision Guide

| Scenario | Recommended Pattern |
|----------|--------------------|
| Small result for plotting / export | `toPandas()` after Spark aggregation |
| Local lookup table into Spark | `spark.createDataFrame(pdf)` |
| Familiar pandas syntax at scale | `pyspark.pandas` (`ps.DataFrame`) |
| Custom column transform | `@pandas_udf` (vectorised) |
| Per-group analytics (z-score, rolling) | `groupBy().applyInPandas()` |
| Batch-wise row transforms | `mapInPandas` |
| Cross-dataset per-key logic | `cogroup().applyInPandas()` |
| Per-group ML preprocessing | `groupBy().applyInPandas()` pipeline |

In [ ]:
spark.stop()